In [1]:
import sys
import argparse
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import numpy as np
import anndata
import statsmodels.api as sm
import statsmodels.formula.api as smf
import tqdm
import pandas as pd



import anndata
import csv
import gzip
import os
import scipy.io
import numpy as np 
import anndata

import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import scanpy
from matplotlib.pyplot import rc_context




In [2]:

# --- INIT ---

# which data to use
head_folder = "/".join(os.getcwd().split("/")[:3])
data_head_folder = '%s/Dropbox/aorta_circadian_data/datasets/joint' % head_folder
hvg_to_use = 'transformed_X_outlier_variance'

# get corresponding paths
adata_path = '%s/adata_qc_filtered.h5ad' % data_head_folder
hvg_folder = '%s/data_annotations/hvg/%s' % (data_head_folder,hvg_to_use)
scvi_res_folder = '%s/scvi_res' % hvg_folder
scvi_mean_embedding_df_path = '%s/scvi_mean_embedding.tsv' % scvi_res_folder
umap_path_out = '%s/scvi_mean_umap_embedding.tsv' % scvi_res_folder
cluster_resolution = 0.05
clustering_folder = '%s/clustering/res_%s' % (scvi_res_folder, str(cluster_resolution))
cluster_df_fileout = '%s/clusters.tsv' % (clustering_folder)


# # SMC subclusters
# clustering_folder = '%s/Dropbox/aorta_circadian_data/datasets/joint/data_annotations/hvg/transformed_X_outlier_variance/scvi_res/clustering/res_0.05/subclustering/res_0.5' % head_folder
# cluster_df_fileout = '%s/smc_subclusters.tsv' % (clustering_folder)



# settings
min_prop = 1e-7




In [3]:
# --- LOAD ADATA ---

adata = anndata.read_h5ad(adata_path)

adata


AnnData object with n_obs × n_vars = 145271 × 32285
    obs: 'sample_id', 'within_experiment_batch_id', 'lib_size', 'sex', 'misaligned', 'zt', 'bmal1_ko', 'batch', 'percent_mito', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mito_qc_pass', 'umi_qc_pass', 'doublet_qc_pass', 'doublet_score', 'qc_pass', 'concat_batch', 'description'
    var: 'ensembl_id', 'n_cells_by_counts-0', 'mean_counts-0', 'log1p_mean_counts-0', 'pct_dropout_by_counts-0', 'total_counts-0', 'log1p_total_counts-0', 'n_cells_by_counts-1', 'mean_counts-1', 'log1p_mean_counts-1', 'pct_dropout_by_counts-1', 'total_counts-1', 'log1p_total_counts-1', 'n_cells_by_counts-2', 'mean_counts-2', 'log1p_mean_counts-2', 'pct_dropout_by_counts-2', 'total_counts-2', 'log1p_total_counts-2'

In [4]:
# --- ADD EMBEDDINGS TO ADATA ---



# ** load clusters **
cluster_df = pd.read_table(cluster_df_fileout,sep='\t',index_col='index')

# ** make sure everything in the same order
adata = adata[list(cluster_df.index)]


# ** add embeddings **
adata.obs["cluster"] = np.array(cluster_df['leiden_scvi_cluster'])
# adata.obs["cluster"] = np.array(cluster_df['smc_subcluster'])

adata


/anaconda2/envs/scrublet/lib/python3.8/site-packages/pandas/core/arrays/categorical.py:2487: FutureWarning: The `inplace` parameter in pandas.Categorical.remove_unused_categories is deprecated and will be removed in a future version.
  res = method(*args, **kwargs)
Trying to set attribute `.obs` of view, copying.


AnnData object with n_obs × n_vars = 145271 × 32285
    obs: 'sample_id', 'within_experiment_batch_id', 'lib_size', 'sex', 'misaligned', 'zt', 'bmal1_ko', 'batch', 'percent_mito', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mito_qc_pass', 'umi_qc_pass', 'doublet_qc_pass', 'doublet_score', 'qc_pass', 'concat_batch', 'description', 'cluster'
    var: 'ensembl_id', 'n_cells_by_counts-0', 'mean_counts-0', 'log1p_mean_counts-0', 'pct_dropout_by_counts-0', 'total_counts-0', 'log1p_total_counts-0', 'n_cells_by_counts-1', 'mean_counts-1', 'log1p_mean_counts-1', 'pct_dropout_by_counts-1', 'total_counts-1', 'log1p_total_counts-1', 'n_cells_by_counts-2', 'mean_counts-2', 'log1p_mean_counts-2', 'pct_dropout_by_counts-2', 'total_counts-2', 'log1p_total_counts-2'

In [5]:
# --- GET THE UNIQUE DESCRIPTION AND CLUSTERS ---

descriptions = list(adata.obs['description'].unique())
clusters = sorted(list(adata.obs['cluster'].unique()))
clusters = list(filter(lambda x: ~np.isnan(x),clusters)) # get rid of NaN cluster (only relevant for SMC subcluster)
clusters = list(map(lambda x: int(x),clusters))


print("Unique descriptions:\n",descriptions)
print("Unique clusters:\n",clusters)


# --- LIMIT ADATA TO GENES THAT MEET MINIMUM PSEUDOBULK COUNT THRESHOLD ---


# ** get gene pseudobulk counts
adata.var['pseudobulk_count'] = np.array(np.sum(adata.X,axis=0)).flatten()

# ** get the number of cells in the smallest cell type
smallest_cell_type_adata = adata[adata.obs['cluster'] == np.max(clusters)]

# ** get pseudobulk cutoff **
pseudobulk_threshold = min_prop * np.sum(smallest_cell_type_adata.obs['lib_size'])

# ** limit adata to this **
adata = adata[:,adata.var['pseudobulk_count'] >= pseudobulk_threshold]


adata


Unique descriptions:
 ['male aligned bmal1-ko', 'male misaligned bmal1-control', 'female aligned bmal1-ko', 'female misaligned bmal1-control', 'male aligned bmal1-control', 'female aligned bmal1-control']
Unique clusters:
 [0, 1, 2, 3, 4, 5, 6]


/anaconda2/envs/scrublet/lib/python3.8/site-packages/pandas/core/arrays/categorical.py:2487: FutureWarning: The `inplace` parameter in pandas.Categorical.remove_unused_categories is deprecated and will be removed in a future version.
  res = method(*args, **kwargs)
/anaconda2/envs/scrublet/lib/python3.8/site-packages/pandas/core/arrays/categorical.py:2487: FutureWarning: The `inplace` parameter in pandas.Categorical.remove_unused_categories is deprecated and will be removed in a future version.
  res = method(*args, **kwargs)


View of AnnData object with n_obs × n_vars = 145271 × 26120
    obs: 'sample_id', 'within_experiment_batch_id', 'lib_size', 'sex', 'misaligned', 'zt', 'bmal1_ko', 'batch', 'percent_mito', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mito_qc_pass', 'umi_qc_pass', 'doublet_qc_pass', 'doublet_score', 'qc_pass', 'concat_batch', 'description', 'cluster'
    var: 'ensembl_id', 'n_cells_by_counts-0', 'mean_counts-0', 'log1p_mean_counts-0', 'pct_dropout_by_counts-0', 'total_counts-0', 'log1p_total_counts-0', 'n_cells_by_counts-1', 'mean_counts-1', 'log1p_mean_counts-1', 'pct_dropout_by_counts-1', 'total_counts-1', 'log1p_total_counts-1', 'n_cells_by_counts-2', 'mean_counts-2', 'log1p_mean_counts-2', 'pct_dropout_by_counts-2', 'total_counts-2', 'log1p_total_counts-2', 'pseudobulk_count'

In [7]:
# --- ADD PHASE COL ---

adata.obs['phase'] = np.array((adata.obs['zt'] / 24.0) * 2 * np.pi)



Trying to set attribute `.obs` of view, copying.


In [6]:
# --- IDENTIFY GENES MEETING THE MINIMUM PROP IN EACH CLUSTER / CONDITION COMBO ---

import warnings
warnings.filterwarnings('ignore')

genes_to_est = set()
for i, description in enumerate(descriptions):
    for j, cluster in enumerate(clusters):
        print("Description: %s; Cluster: %s" % (i,j))
            
        cluster_condition_adata = adata[(adata.obs['cluster'] == cluster) & (adata.obs['description'] == description)]
        cluster_condition_adata.var['prop'] = np.array(np.sum(cluster_condition_adata.X,axis=0)).flatten()  / np.sum(cluster_condition_adata.obs['lib_size'])
        cluster_condition_adata = cluster_condition_adata[:,cluster_condition_adata.var['prop'] >= min_prop]
        genes_to_est.update(list(cluster_condition_adata.var_names))

# make it a list
genes_to_est = list(genes_to_est)



Description: 0; Cluster: 0


Trying to set attribute `.var` of view, copying.
Trying to set attribute `.var` of view, copying.


Description: 0; Cluster: 1
Description: 0; Cluster: 2


Trying to set attribute `.var` of view, copying.
Trying to set attribute `.var` of view, copying.
Trying to set attribute `.var` of view, copying.
Trying to set attribute `.var` of view, copying.
Trying to set attribute `.var` of view, copying.


Description: 0; Cluster: 3
Description: 0; Cluster: 4
Description: 0; Cluster: 5
Description: 0; Cluster: 6
Description: 1; Cluster: 0


...

Description: 4; Cluster: 2
Description: 4; Cluster: 3
Description: 4; Cluster: 4
Description: 4; Cluster: 5
Description: 4; Cluster: 6
Description: 5; Cluster: 0


Trying to set attribute `.var` of view, copying.
Trying to set attribute `.var` of view, copying.
Trying to set attribute `.var` of view, copying.


Description: 5; Cluster: 1
Description: 5; Cluster: 2


Trying to set attribute `.var` of view, copying.
Trying to set attribute `.var` of view, copying.
Trying to set attribute `.var` of view, copying.
Trying to set attribute `.var` of view, copying.


Description: 5; Cluster: 3
Description: 5; Cluster: 4
Description: 5; Cluster: 5
Description: 5; Cluster: 6


In [8]:
# --- GET THE OLD AND NEW FOLDER OUTS ---

# ** old **
old_reg_head_folder = '%s/nonparametric_reg_old' % clustering_folder
old_cluster_description_folder_out_dict = {}
for cluster in clusters:
    cluster_reg_folder = '%s/cluster_%s' % (old_reg_head_folder,cluster)
    for description in descriptions:
        cluster_description_reg_folder = '%s/%s' % (cluster_reg_folder,description)
        if cluster not in old_cluster_description_folder_out_dict:
            old_cluster_description_folder_out_dict[cluster] = {}
        old_cluster_description_folder_out_dict[cluster][description] = cluster_description_reg_folder

    
# ** new **
new_reg_head_folder = '%s/nonparametric_reg' % clustering_folder
new_cluster_description_folder_out_dict = {}
for cluster in clusters:
    cluster_reg_folder = '%s/cluster_%s' % (new_reg_head_folder,cluster)
    for description in descriptions:
        cluster_description_reg_folder = '%s/%s' % (cluster_reg_folder,description)
        if cluster not in new_cluster_description_folder_out_dict:
            new_cluster_description_folder_out_dict[cluster] = {}
        new_cluster_description_folder_out_dict[cluster][description] = cluster_description_reg_folder
    
    
    

In [62]:
# --- FOR EACH SMC, LOOK AT MISSING GENES ---

cluster = 0
for i, description in enumerate(descriptions):
    
    # ** load the old and new regression **
    old_file = '%s/de_novo_metrics.tsv' % old_cluster_description_folder_out_dict[cluster][description]
    new_file = '%s/de_novo_metrics.tsv' % new_cluster_description_folder_out_dict[cluster][description]
    old_df = pd.read_table(old_file,sep='\t',index_col='gene')
    new_df = pd.read_table(new_file,sep='\t',index_col='gene')
    
    # ** load the estimated and unestimated gene set **
    estimated_gene_set = set(old_df.index).union(set(new_df.index))
    unestimated_gene_set = set(genes_to_est).difference(estimated_gene_set)
    unique_old_gene_set = set(old_df.index).difference(set(new_df.index))
    unique_new_gene_set = set(new_df.index).difference(set(old_df.index))
    intersection_gene_set = set(old_df.index).intersection(set(new_df.index))

    
#     # ** get the concat df for each set of parameters and write out **
#     file_list = ['de_novo_metrics.tsv','gene_log_alpha.tsv','gene_log_beta.tsv','log_min_max.tsv']
#     for f in file_list:
        
#         # get concat df
#         old_path = '%s/%s' % (old_cluster_description_folder_out_dict[cluster][description],f)
#         new_path = '%s/%s' % (new_cluster_description_folder_out_dict[cluster][description],f)
#         old_df = pd.read_table(old_path,sep='\t',index_col='gene')
#         new_df = pd.read_table(new_path,sep='\t',index_col='gene')
#         concat_df = pd.concat((old_df.loc[unique_old_gene_set],old_df.loc[intersection_gene_set],new_df.loc[unique_new_gene_set]))
        
#         # write out
#         concat_df.to_csv(new_path,sep='\t')

        
        
    
    
    # ** get cluster, condition adata **
    cluster_condition_adata = adata[(adata.obs['cluster'] == cluster) & (adata.obs['description'] == description)]
    cluster_condition_adata = cluster_condition_adata[:,list(unestimated_gene_set)]

    # ** prep **
    cluster_condition_adata.var['prop'] = np.array(np.sum(cluster_condition_adata.X,axis=0)).flatten() / np.sum(cluster_condition_adata.obs['lib_size'])
    if 'log_L' not in adata.obs:
        cluster_condition_adata.obs['log_L'] = np.array(np.log(cluster_condition_adata.obs['lib_size']))

    # ** check how many gene unestimated had prop's greater than 0 **
    unestimated_adata = cluster_condition_adata[:,cluster_condition_adata.var['prop'] > 0.0]
    
    # ** get the new unestimated gene set (with prop's greater than 0) **
    unestimated_genes = list(unestimated_adata.var_names)
    
    print(len(unestimated_genes))
    
    


Trying to set attribute `.var` of view, copying.


0


Trying to set attribute `.var` of view, copying.


0


Trying to set attribute `.var` of view, copying.


...

0


Trying to set attribute `.var` of view, copying.


0


Trying to set attribute `.var` of view, copying.


0
